# RegDrift: LoRA-tune the obligation extractor
**Runtime > Change runtime type > T4 GPU** first.
Trains a small LoRA adapter on `data/train.jsonl` (seed + reviewer feedback). 13 seed rows only prove the pipeline; add 40+ real clauses for a meaningful model.

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/RegDrift.git"  # change this
!git clone $REPO_URL repo
%cd repo
!pip install -q transformers peft datasets accelerate
!python build_lora_data.py

In [ ]:
import json, torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model

BASE = "Qwen/Qwen2.5-0.5B-Instruct"   # small, ungated, fits free T4
tok = AutoTokenizer.from_pretrained(BASE)
if tok.pad_token is None: tok.pad_token = tok.eos_token
rows = [json.loads(l) for l in open("data/train.jsonl")]
ds = Dataset.from_list([{"text": r["text"] + tok.eos_token} for r in rows])
ds = ds.map(lambda b: tok(b["text"], truncation=True, max_length=384), remove_columns=["text"])

model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float32).to("cuda")
model = get_peft_model(model, LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05,
        target_modules=["q_proj","v_proj"], task_type="CAUSAL_LM"))
model.print_trainable_parameters()

args = TrainingArguments(output_dir="out", num_train_epochs=8, per_device_train_batch_size=4,
        learning_rate=2e-4, logging_steps=5, save_strategy="no", fp16=True, report_to=[])
Trainer(model=model, args=args, train_dataset=ds,
        data_collator=DataCollatorForLanguageModeling(tok, mlm=False)).train()
model.save_pretrained("regdrift-lora"); tok.save_pretrained("regdrift-lora")

In [ ]:
# Quick test on an unseen clause
P = "### Instruction:\nExtract measurable obligations as a JSON list.\n### Clause:\n{c}\n### Output:\n"
c = "Customer names must be checked against sanctions lists every 3 days."
ids = tok(P.format(c=c), return_tensors="pt").to("cuda")
out = model.generate(**ids, max_new_tokens=120, do_sample=False)
print(tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True))

In [ ]:
# Download the adapter, then commit it to your repo (small) or upload to Hugging Face
!zip -qr regdrift-lora.zip regdrift-lora
from google.colab import files; files.download("regdrift-lora.zip")

## Use it
In the run notebook / Actions: unzip to `regdrift-lora/`, `pip install transformers peft torch`, set `LORA_ADAPTER=regdrift-lora`, then `python regdrift.py ...`. Extraction will use your tuned model, falling back to Groq/rules if it fails.